## experiments
Observe how LLR changes with the size of an arbitrary sphere of data.

## conclusion
Changing the size of the region yields different LLR (with no effect imposed)

note that slope of line (llr increase or decrease with size) varies with image set.  on `exp_monkey` below, the slope of this line increases with decreased `noise_scale` (more consistent images across population -> larger increase in llr with every included voxel ... consider extereme example: one is given N repeats of the same image.  This N artificially inflates "repeated observations" to increase LLR)

In [ ]:
import hglm
import pathlib

# prep hcp dataset
folder = '/home/matt/Dropbox/pnl_hglm/data/hcp100_lowres/image'
exp_hcp = hglm.ExperimentImageOnly.from_search(folder=folder,
                                               sbj_regex='[\d]{6}',
                                               img_glob_dict={'FA': '*_FA.nii.gz',
                                                              'MD': '*_MD.nii.gz'})
exp_hcp = exp_hcp.sample_x(a=2, seed=1)

# prep mandrill dataset
seed = 0
num_img = 5
noise_scale = .1

folder = pathlib.Path(hglm.__file__).parents[1] / 'test' / 'data'
exp_monkey = hglm.ExperimentImageOnly.from_search(folder=folder, 
                                           sbj_regex='mandrill_small', 
                                           img_glob_dict={'rgb': 'mandrill_small.png'})
exp_monkey.bootstrap_img(n=num_img, seed=seed, noise_scale=noise_scale)
exp_monkey = exp_monkey.sample_x(a=2, seed=seed)

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed
from itertools import product

p_val = .4
effect_perc = .2

num_seed = 100
radius_all = np.arange(1, 19, 2)
exp = exp_monkey

# parallel (all threads)
n_jobs = 1

# prep
get_eps = hglm.experiment.prep_get_eps(np.atleast_2d(exp.x[0, :])), \
          hglm.experiment.prep_get_eps(exp.x)

exp = hglm.ExperimentWhitened.from_exp(exp)

def run(exp, seed, radius):
    # sample a subset of voxels (sphere of given radius), constrain experiment
    extenter = hglm.extent.ExtenterSphere(radius=radius)
    mask = extenter(mask_idx=exp.mask_idx, seed=seed)        
    _exp = exp.apply_mask(mask)

    # # sample & apply effect effect
    # n = exp.y.shape[2] * effect_perc
    # extenter = hglm.ExtenterMinVar(n=n)
    # mask_target = extenter(y=exp.y, mask_idx=exp.mask_idx, seed=seed)
    # _exp, effect = _exp.impose_effect(mask=mask_target, p_val=p_val)


    # compute llr
    size = mask.sum()
    _y = _exp.y.reshape((_exp.y.shape[0], -1), order='F')
    yout = _y @ _y.T
    ybar  = _exp.y.mean(axis=2)
    eps = tuple(fnc(size=size, yout=yout, ybar=ybar) for fnc in get_eps)
    llr = hglm.experiment.get_llr(size=size, eps0=eps[0], eps1=eps[1])

    # compute sigma
    sigma = hglm.experiment.get_sigma(size, yout, ybar)

    # compute percent error which is sigma
    perc_sigma = tuple(np.trace(sigma) / np.trace(e) for e in eps)

    return dict(llr=llr, seed=seed, radius=radius, size=size, tr_sigma=np.trace(sigma), tr_eps0=np.trace(eps[0]), tr_eps1=np.trace(eps[1]), perc_sigma0=perc_sigma[0], perc_sigma1=perc_sigma[1])

d_list = list()

arg_iter = product((exp, ), range(num_seed), radius_all)
arg_iter = tqdm(arg_iter, total=num_seed * radius_all.size)
if n_jobs not in (0, 1):
    r = Parallel(n_jobs=n_jobs, verbose=10)(delayed(run)(*args) for args in arg_iter)
    df = pd.DataFrame(r)
else:
    d_list = list()
    for args in arg_iter:
        d_list.append(run(*args))
    df = pd.DataFrame(d_list)

In [ ]:
df.head()

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

y_feat = 'tr_eps0'

x = np.ones((df.shape[0], 2))
x[:, 1] = np.log(df['size'].values)
y = np.log(df[y_feat].values)
beta = np.linalg.pinv(x) @ y
print(beta)
y_hat = np.exp(x @ beta)

fig1 = px.scatter(df, x='size', y=y_feat, hover_data=['seed', 'radius'], color='seed')
fig2 = px.line(x=df['size'].values, y=y_hat)

fig = go.Figure(data=fig1.data + fig2.data)
fig.update_xaxes(type='log')
fig.update_yaxes(type='log')